# Surface Classifier — Colab Training Notebook

Trains MobileNetV3-Small on RealSense ground patches (carpet/tile/grass/...) and exports ONNX.

**Prerequisites**
- The `labs/` repo lives in your Google Drive (it already does if you work from the Mac's Google Drive mount).
- Collected patches organized as `<DRIVE_ROOT>/<DATA_SUBDIR>/<class_name>/*.png` — e.g. `labs/misc/carpet/*.png`, `labs/misc/tile/*.png`.
- **Runtime → Change runtime type → T4 GPU** (free tier is enough).

**Expected runtime** on a T4 for ~335 patches / 30 epochs: ~5 min end-to-end (mount + sync + train + export).

**Outputs** land back on Drive at `<DRIVE_ROOT>/lab-7-vision-lab-team3/surface_classifier/models/` and sync to your Mac within ~1 min.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config
Edit these variables — everything else is wired through them.

In [ ]:
import os

# --- Paths on Drive (mount point is /content/drive/MyDrive/) ---
DRIVE_ROOT = '/content/drive/MyDrive/Acad/Penn/sem2-spr26/f110/labs'
DATA_SUBDIR = 'misc'                                  # <DRIVE_ROOT>/<DATA_SUBDIR>/<label>/*.png
PKG_SUBDIR  = 'lab-7-vision-lab-team3/surface_classifier'

# --- Local Colab workspace (fast I/O, scratch) ---
WORK_DIR = '/content/surface_classifier'

# --- Hyperparameters ---
EPOCHS       = 30
BATCH_SIZE   = 32
INPUT_SIZE   = 224
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
VAL_FRAC     = 0.2
SEED         = 0
NUM_WORKERS  = 2

# Derived paths
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/{DATA_SUBDIR}'
DRIVE_PKG_DIR  = f'{DRIVE_ROOT}/{PKG_SUBDIR}'
DRIVE_MODELS   = f'{DRIVE_PKG_DIR}/models'

assert os.path.isdir(DRIVE_DATA_DIR), f'Data folder not found: {DRIVE_DATA_DIR}'
assert os.path.isdir(DRIVE_PKG_DIR),  f'Package folder not found: {DRIVE_PKG_DIR}'
print('Data   :', DRIVE_DATA_DIR)
print('Package:', DRIVE_PKG_DIR)
print('Work   :', WORK_DIR)

Data   : /content/drive/MyDrive/Acad/Penn/sem2-spr26/f110/labs/misc
Package: /content/drive/MyDrive/Acad/Penn/sem2-spr26/f110/labs/lab-7-vision-lab-team3/surface_classifier
Work   : /content/surface_classifier


## 3. GPU sanity check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — go to Runtime → Change runtime type → T4 GPU'

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
torch 2.10.0+cu128 | cuda available: True


## 4. Sync Drive → local Colab storage
Drive I/O inside Colab is slow (~1–5 MB/s). We copy the dataset and package code to local disk once and train from there.

In [ ]:
!mkdir -p {WORK_DIR}/data {WORK_DIR}/models
# Copy dataset (PNGs only)
!rsync -a --include='*/' --include='*.png' --exclude='*' "{DRIVE_DATA_DIR}/" "{WORK_DIR}/data/"
# Copy scripts from the package
!rsync -a "{DRIVE_PKG_DIR}/scripts/" "{WORK_DIR}/scripts/"
!ls {WORK_DIR}/data && echo --- && find {WORK_DIR}/data -name '*.png' | awk -F/ '{{print $(NF-1)}}' | sort | uniq -c

carpet	cloth  sticky  tile
---
    168 carpet
    152 cloth
    150 sticky
    329 tile


## 5. Install any missing dependencies
Colab T4 images ship with torch / torchvision / opencv-python / numpy / pyyaml already. This cell is a safety net.

In [ ]:
import importlib, subprocess, sys

needed = {
    'torch':       'torch',
    'torchvision': 'torchvision',
    'cv2':         'opencv-python',
    'yaml':        'pyyaml',
    'numpy':       'numpy',
    # ONNX-side deps for export_onnx.py. torch.onnx.export needs `onnx` for
    # serialization (and `onnxscript` as a graph backend on torch >= 2.9)
    # even when dynamo=False.
    'onnx':       'onnx',
    'onnxscript': 'onnxscript',
}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All deps present.')

Installing: ['onnx', 'onnxscript']


## 6. Build train/val split

In [ ]:
%cd {WORK_DIR}
!python scripts/make_split.py --root data --out-dir . --val-frac {VAL_FRAC} --seed {SEED}
!head -3 manifest_train.csv && echo --- && cat classes.txt

/content/surface_classifier
      carpet: 168 total, 34 val, 134 train
       cloth: 152 total, 30 val, 122 train
      sticky: 150 total, 30 val, 120 train
        tile: 329 total, 66 val, 263 train
wrote ./manifest_train.csv (639 rows)
wrote ./manifest_val.csv (160 rows)
wrote ./classes.txt (4 classes)
path,label
data/tile/1777138932.898.png,tile
data/carpet/1777141369.082.png,carpet
---
carpet
cloth
sticky
tile


## 7. Train
`train.py` auto-selects CUDA when available.

In [8]:
%cd {WORK_DIR}
!python scripts/train.py \
    --train-manifest manifest_train.csv \
    --val-manifest   manifest_val.csv   \
    --classes        classes.txt        \
    --out            models/surface_mnv3.pt \
    --input-size     {INPUT_SIZE}       \
    --epochs         {EPOCHS}           \
    --batch-size     {BATCH_SIZE}       \
    --lr-head        {LR_HEAD}          \
    --lr-backbone    {LR_BACKBONE}      \
    --num-workers    {NUM_WORKERS}

/content/surface_classifier
[device] cuda
[classes] ['carpet', 'cloth', 'sticky', 'tile']
[data] train=639 val=160
[class weights] {'carpet': 1.0739600658416748, 'cloth': 1.1795954704284668, 'sticky': 1.1992555856704712, 'tile': 0.5471888184547424}
epoch 01/30 | train_loss=0.5108 train_acc=0.836 val_acc=0.688
  ↑ saved models/surface_mnv3.pt (val_acc=0.688)
epoch 02/30 | train_loss=0.1901 train_acc=0.928 val_acc=0.800
  ↑ saved models/surface_mnv3.pt (val_acc=0.800)
epoch 03/30 | train_loss=0.0798 train_acc=0.967 val_acc=0.919
  ↑ saved models/surface_mnv3.pt (val_acc=0.919)
epoch 04/30 | train_loss=0.0812 train_acc=0.973 val_acc=0.900
epoch 05/30 | train_loss=0.0546 train_acc=0.977 val_acc=0.912
epoch 06/30 | train_loss=0.0355 train_acc=0.986 val_acc=0.938
  ↑ saved models/surface_mnv3.pt (val_acc=0.938)
epoch 07/30 | train_loss=0.0366 train_acc=0.983 val_acc=0.950
  ↑ saved models/surface_mnv3.pt (val_acc=0.950)
epoch 08/30 | train_loss=0.0341 train_acc=0.987 val_acc=0.956
  ↑ saved 

## 8. Export ONNX
Single-file (legacy exporter, `dynamo=False`) — portable to the Jetson.

In [9]:
%cd {WORK_DIR}
!python scripts/export_onnx.py --ckpt models/surface_mnv3.pt --out models/surface_mnv3.onnx
!ls -lh models/

/content/surface_classifier
[load] models/surface_mnv3.pt | classes=['carpet', 'cloth', 'sticky', 'tile'] | input_size=224
/content/surface_classifier/scripts/export_onnx.py:37: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
[export] models/surface_mnv3.onnx
total 12M
-rw-r--r-- 1 root root   25 Apr 25 18:54 classes.txt
-rw-r--r-- 1 root root 5.9M Apr 25 18:56 surface_mnv3.onnx
-rw-r--r-- 1 root root 6.0M Apr 25 18:54 surface_mnv3.pt


## 9. Sync local → Drive
Writes `.pt`, `.onnx`, `classes.txt`, and the manifests back to the package on Drive. Your Mac will pick them up within a minute via Drive sync.

In [10]:
!mkdir -p "{DRIVE_MODELS}"
!rsync -a "{WORK_DIR}/models/" "{DRIVE_MODELS}/"
# Manifests are training artifacts — also useful to keep on Drive for reproducibility.
!cp "{WORK_DIR}/manifest_train.csv" "{WORK_DIR}/manifest_val.csv" "{WORK_DIR}/classes.txt" "{DRIVE_PKG_DIR}/"
!ls -lh "{DRIVE_MODELS}"

total 16M
-rw------- 1 root root   25 Apr 25 18:54 classes.txt
-rw------- 1 root root 3.8M Apr 23 23:55 surface_mnv3_fp16.trt
-rw------- 1 root root 5.9M Apr 25 18:56 surface_mnv3.onnx
-rw------- 1 root root 6.0M Apr 25 18:54 surface_mnv3.pt


## 10. (Optional) Confusion matrix + misclassified grid
Loads the best checkpoint, runs val-set inference, prints the confusion matrix, and displays misclassified patches.

In [11]:
import csv
import numpy as np
import torch
from torch import nn
from torchvision import models, transforms
import cv2
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(f'{WORK_DIR}/models/surface_mnv3.pt', map_location=device, weights_only=False)
classes = ckpt['classes']
input_size = ckpt['input_size']

m = models.mobilenet_v3_small(weights=None)
m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, len(classes))
m.load_state_dict(ckpt['model'])
m.eval().to(device)

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(int(input_size * 1.15)),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
cls_to_idx = {c: i for i, c in enumerate(classes)}

rows = []
with open(f'{WORK_DIR}/manifest_val.csv') as f:
    for r in csv.DictReader(f):
        rows.append((r['path'], r['label']))

cm = np.zeros((len(classes), len(classes)), dtype=int)
mistakes = []  # (path, true, pred, prob_pred)
with torch.no_grad():
    for path, label in rows:
        img = cv2.imread(path)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        x = val_tf(img).unsqueeze(0).to(device)
        probs = torch.softmax(m(x), dim=1).cpu().numpy().reshape(-1)
        pred = int(np.argmax(probs))
        true = cls_to_idx[label]
        cm[true, pred] += 1
        if pred != true:
            mistakes.append((path, label, classes[pred], float(probs[pred])))

print('Classes:', classes)
print('Confusion matrix (rows=true, cols=pred):')
print(cm)
acc = np.trace(cm) / cm.sum()
print(f'Val accuracy: {acc:.3f}   Mistakes: {len(mistakes)}/{cm.sum()}')

# Misclassified grid
if mistakes:
    n = min(12, len(mistakes))
    cols = 4
    rows_ = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 3, rows_ * 3))
    for i, (p, tr, pr, c) in enumerate(mistakes[:n]):
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(rows_, cols, i + 1)
        plt.imshow(img)
        plt.title(f'true={tr}\npred={pr} ({c:.2f})', fontsize=9)
        plt.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No mistakes on val set.')

Classes: ['carpet', 'cloth', 'sticky', 'tile']
Confusion matrix (rows=true, cols=pred):
[[34  0  0  0]
 [ 0 30  0  0]
 [ 0  0 30  0]
 [ 0  0  0 66]]
Val accuracy: 1.000   Mistakes: 0/160
No mistakes on val set.
